In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv(r'D:\AQI Project\dataset\final_valley_dataset.csv')

In [ ]:
print(data.isna().sum())

In [ ]:
missing_values = data.isna().sum()/ len(data)
print(missing_values[missing_values > 0])

In [ ]:
import missingno as msno
msno.matrix(data)

In [ ]:
msno.heatmap(data)

In [ ]:
msno.dendrogram(data)

In [ ]:
missing_values = data[data['Station'] == 'Bhaktapur'][['PM2_5_ugm3','PM10_ugm3']].isnull().sum()/ len(data)
missing_values2 = data[data['Station'] == 'Bhaisipati'][['PM2_5_ugm3','PM10_ugm3']].isnull().sum()/ len(data)
missing_values3 = data[data['Station'] == 'Dhulikhel'][['PM2_5_ugm3','PM10_ugm3']].isnull().sum()/ len(data)
missing_values4 = data[data['Station'] == 'Pulchowk'][['PM2_5_ugm3','PM10_ugm3']].isnull().sum()/ len(data)
missing_values5 = data[data['Station'] == 'Ratnapark'][['PM2_5_ugm3','PM10_ugm3']].isnull().sum()/ len(data)
missing_values6 = data[data['Station'] == 'Shankhapark'][['PM2_5_ugm3','PM10_ugm3']].isnull().sum()/ len(data)
missing_values7 = data[data['Station'] == 'TU / Kirtipur'][['PM2_5_ugm3','PM10_ugm3']].isnull().sum()/ len(data)

print(f'Bhaktapur : {missing_values[missing_values > 0]}')
print()
print(f'Bhaisipati : {missing_values2[missing_values2 > 0]}')
print()
print(f'Dhulikhel : {missing_values3[missing_values3 > 0]}')
print()
print(f'Pulchowk : {missing_values4[missing_values4 > 0]}')
print()
print(f'Ratnapark : {missing_values5[missing_values5 > 0]}')
print()
print(f'Shankhapark : {missing_values6[missing_values6 > 0]}')
print()
print(f'TU/Kirtipur : {missing_values7[missing_values7 > 0]}')


In [ ]:
import seaborn as sns

sns.set_theme(style = 'darkgrid')
sns.set_palette(sns.color_palette('Set1'))

In [ ]:
data = data.drop(columns= ['Unnamed: 0' , 'elevation_m' , 'Region' , 'era5_rh_pct' , 'era5_wind_speed_ms' , 'era5_wind_dir_deg' , 'ground_precip_mm' , 'ground_tmin_c' ,'ground_tmax_c' , 'ground_precip_src_index','ground_precip_dist_km','ground_temp_src_index','ground_temp_dist_km'])

In [ ]:
missingvalues2 = data.isna().sum() / len(data)

print(missingvalues2[missingvalues2 > 0])

In [ ]:
data = data.dropna(subset = ['era5_t2m_c' , 'era5_tmin_c' , 'era5_tmax_c' , 'era5_dewpoint_c' , 'era5_wind_u_ms' , 'era5_wind_v_ms' , 'era5_surface_pressure_hpa' , 'era5_precip_mm' , 'era5_solar_rad_mj_m2' ])

In [ ]:
import numpy as np

In [ ]:
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values(by = ['Station' , 'Date']).reset_index(drop = True)

In [ ]:
min_date = data['Date'].min()
data['day_index'] = (data['Date'] - min_date).dt.days


data['day_of_year'] = data['Date'].dt.dayofyear
data['sin_day'] = np.sin(2*np.pi* data['day_of_year'] / 365.25)
data['cos_day'] = np.cos(2*np.pi* data['day_of_year'] / 365.25)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

In [ ]:
data.head()

In [ ]:
print(data.columns.unique())

In [ ]:
feature_columns = [ 'era5_t2m_c', 'era5_tmax_c',
       'era5_tmin_c', 'era5_dewpoint_c', 'era5_wind_u_ms', 'era5_wind_v_ms',
       'era5_surface_pressure_hpa', 'era5_precip_mm', 'era5_solar_rad_mj_m2',
        'sin_day', 'cos_day']

In [ ]:
print(type(data))

In [ ]:
scaler = StandardScaler()

X_scaled = pd.DataFrame (scaler.fit_transform(data[feature_columns]) , columns = feature_columns) 


In [ ]:
tier1_pm_columns = ['PM2_5_ugm3', 'PM10_ugm3'] 
tier2_pm_columns = ['TSP_ugm3', 'PM1_generic_ugm3']

target_weights = {}

In [ ]:
idx_tier1 = data.dropna(subset = tier1_pm_columns).index

X_train_tier1 = X_scaled.loc[idx_tier1]


print('Training for tier 1 columns importance')


for col in tier1_pm_columns : 

    y_train = data.loc[idx_tier1 , col]

    rf = RandomForestRegressor(n_estimators= 100 , random_state= 42 , n_jobs = -1)

    rf.fit(X_train_tier1 , y_train)

    target_weights[col] = dict(zip(feature_columns , rf.feature_importances_))

In [ ]:
all_columns = tier1_pm_columns + tier2_pm_columns

idx_tier2 = data.dropna(subset = tier1_pm_columns + tier2_pm_columns).index

X_train_tier2 = X_scaled.loc[idx_tier2]

for col in all_columns : 

    rf2 = RandomForestRegressor(n_estimators= 100 , random_state= 42 , n_jobs = -1)
    
    rf2.fit(X_train_tier2 , data.loc[idx_tier2 , col])

    target_weights[col] = dict(zip(feature_columns , rf2.feature_importances_))

In [ ]:
for key , value in target_weights.items() :

    print(f'{key} : {value}')
    print()                   

In [ ]:
def haversine_distance(lat1 , lon1 , lat2 , lon2):

    R_earth = 6371.0

    lat1 , lon1 , lat2 , lon2 = map(np.radians , [lat1 , lon1 , lat2 , lon2])

    diff_lat = lat2 - lat1
    diff_lon = lon2 - lon1

    a = np.sin(diff_lat/2.0) **2   + np.cos(lat1) * np.cos(lat2)* np.sin(diff_lon /2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R_earth * c


In [ ]:
station_info = data[['Station' , 'latitude' , 'longitude']].drop_duplicates().set_index('Station')

stations = station_info.index.tolist()

stations

In [ ]:
distance_matrix = pd.DataFrame(index = stations , columns = stations , dtype = float)

for station1 in stations :

    for station2 in stations :

        lat1 , lon1 = station_info.loc[station1 , 'latitude' ] , station_info.loc[station1 , 'longitude']
        lat2 , lon2 = station_info.loc[station2 , 'latitude'] , station_info.loc[station2, 'longitude']


        distance = haversine_distance(lat1, lon1 , lat2 , lon2)
        distance_matrix.loc[station1 , station2] = distance


print(f'Geographic Distance Matrix (km) : \n\n {distance_matrix.round(3)} ')


In [ ]:
pm_pivot = data.pivot_table(index = 'Date' , columns = 'Station' , values = 'PM2_5_ugm3')

correlation_matrix_raw = pm_pivot.corr(method = 'pearson')

corr_matrix = correlation_matrix_raw.clip(lower= 0)

final_corr_matrix = corr_matrix.fillna(0.1)

print(f'Historical Similarity matrix is : \n {final_corr_matrix.round(3)}')


In [ ]:
from wKNN import wKNN
model = wKNN ( k =5 , lambda_decay= 0.2 , rain_threshold= 0.5)

In [ ]:
model.fit(data = data , dist_matrix = distance_matrix, sim_matrix = final_corr_matrix, target_weights = target_weights)